# LAQCC parity (fan-in)

This example uses `parity_laqcc` to fan the XOR parity of four input qubits into one target qubit:

$$|x_0, x_1, x_2, x_3\rangle|t\rangle \mapsto |x_0, x_1, x_2, x_3\rangle|t \oplus x_0 \oplus x_1 \oplus x_2 \oplus x_3\rangle.$$

The input register is preserved. For four or more inputs, the LAQCC construction uses measurement and classical feed-forward to reduce quantum depth.

In [1]:
from typing import no_type_check

from guppylang import guppy
from guppylang.std.builtins import output
from guppylang.std.quantum import collect_measurements, measure, measure_array, qubit, x

from guppyalgos.primitives.subroutines.parity import parity_laqcc
from guppyalgos.utils import qarray

Prepare the input state $|1, 0, 1, 1\rangle$ and target $|0\rangle$. Its input parity is odd, so the target should finish in $|1\rangle$.

In [2]:
input_bits = [True, False, True, True]
n_input_qubits = len(input_bits)
n_ancilla_q = n_input_qubits - 3
total_qubits = n_input_qubits + 1 + 2 * n_ancilla_q

In [3]:
@guppy
@no_type_check
def main() -> None:
    inputs = qarray(n_input_qubits)
    target = qubit()

    bits = input_bits
    for i in range(n_input_qubits):
        if bits[i]:
            x(inputs[i])

    parity_laqcc(target, inputs)

    output("inputs", collect_measurements(measure_array(inputs)))
    output("target", measure(target).read())

In [4]:
result = main.emulator(n_qubits=total_qubits).with_seed(42).with_shots(1).run()
shot = result.collated_shots()[0]
measured_inputs = shot["inputs"][0]
measured_target = shot["target"][0]

print("Input register:", measured_inputs)
print("Parity target:", measured_target)

assert measured_inputs == input_bits
assert measured_target == (sum(input_bits) % 2 == 1)

Input register: [1, 0, 1, 1]
Parity target: 1
